In [1]:
! pip install rdkit
! pip install gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 14.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import rdkit
import gdown

In [3]:
# 1. Get the URL from the 'Share' button in Drive
url = 'https://drive.google.com/file/d/1uVl3dKKGAvyoW_53R-L4abTns3AQfbmy/view?usp=sharing'

# 2. Download the file
# 'fuzzy=True' allows gdown to extract the File ID automatically from the link
output = 'dataset.csv'
gdown.download(url, output, quiet=False, fuzzy=True)

# 3. Load with Pandas
import pandas as pd
df = pd.read_csv(output)

Downloading...
From: https://drive.google.com/uc?id=1uVl3dKKGAvyoW_53R-L4abTns3AQfbmy
To: /content/dataset.csv
100%|██████████| 525k/525k [00:00<00:00, 88.1MB/s]


In [5]:
df.head(5)

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [4]:
df.columns

Index(['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
       'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
       'mol_id', 'smiles'],
      dtype='object')

In [ ]:
df.shape

(7831, 14)

In [ ]:
print(f"total number of null values= {df.isnull().sum().sum()}")

total number of null values= 16026


In [ ]:
df.isnull().sum()

,0
NR-AR,566
NR-AR-LBD,1073
NR-AhR,1282
NR-Aromatase,2010
NR-ER,1638
NR-ER-LBD,876
NR-PPAR-gamma,1381
SR-ARE,1999
SR-ATAD5,759
SR-HSE,1364


In [ ]:
targets = [
'NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase',
'NR-ER','NR-ER-LBD','NR-PPAR-gamma',
'SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53'
]

df[targets] = df[targets].fillna(0)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol)
    ]
features = df['smiles'].apply(smiles_to_features)

# remove invalid rows
df = df[features.notnull()]
features = features[features.notnull()]

X = list(features)

[14:00:39] WARNING: not removing hydrogen atom without neighbors
[14:00:40] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:00:40] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:00:40] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:00:41] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:00:42] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:00:42] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:00:43] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:00:44] Explicit valence for atom # 20 Al, 6, is greater than permitted


In [ ]:
print(X)

[[258.32399999999996, 1.3424, 1, 5, 82.28, 3], [204.22899999999998, 1.2993999999999999, 1, 2, 49.410000000000004, 2], [288.475, 5.090300000000005, 1, 1, 20.23, 1], [276.42400000000004, 3.7524400000000018, 1, 2, 32.34, 7], [206.027, -0.9922000000000002, 5, 3, 135.29000000000002, 2], [290.444, 4.817200000000005, 0, 4, 36.92, 7], [176.62400000000002, 1.6141, 0, 2, 34.14, 1], [621.9340000000001, 4.625400000000002, 2, 3, 66.76, 4], [152.146, -2.9463000000000004, 5, 5, 101.15, 4], [351.8019999999999, 2.1911000000000005, 0, 4, 80.25999999999999, 12], [663.4300000000004, -3.6478999999999964, 7, 17, 321.0900000000001, 11], [354.1, -1.2181000000000002, 3, 5, 104.55000000000001, 2], [144.21399999999997, 1.8416000000000001, 0, 2, 26.3, 3], [198.21799999999996, 1.2249999999999999, 0, 4, 52.6, 5], [215.70799999999997, 0.24750000000000005, 0, 3, 9.72, 2], [185.202, 1.1321999999999994, 0, 2, 40.129999999999995, 2], [231.37999999999997, 2.4633, 2, 3, 55.480000000000004, 12], [178.235, 1.5636, 0, 2, 33.

In [ ]:
feature_names = [
    "MolWt", "LogP", "HDonors", "HAcceptors", "TPSA", "RotBonds"
]

features_df = pd.DataFrame(features.tolist(), columns=feature_names)
features_df.index = df.index
df = pd.concat([df, features_df], axis=1)

In [ ]:
df.columns

Index(['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
       'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
       'mol_id', 'smiles', 'MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA',
       'RotBonds'],
      dtype='object')

In [ ]:
targets = [
    'NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase',
    'NR-ER','NR-ER-LBD','NR-PPAR-gamma',
    'SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53'
]

y = df[targets]

feature_cols = [
    'MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA', 'RotBonds'
]

X = df[feature_cols]

In [ ]:
X.head()

,MolWt,LogP,HDonors,HAcceptors,TPSA,RotBonds
0,258.324,1.34240,1,5,82.28,3
1,204.229,1.29940,1,2,49.41,2
2,288.475,5.09030,1,1,20.23,1
3,276.424,3.75244,1,2,32.34,7
4,206.027,-0.99220,5,3,135.29,2


In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# create generator once
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp = fpgen.GetFingerprint(mol)
    arr = np.array(fp)
    return arr

fps = df['smiles'].apply(smiles_to_fp)

df = df[fps.notnull()]
fps = fps[fps.notnull()]

fp_df = pd.DataFrame(fps.tolist())
fp_df.index = df.index

df = pd.concat([df, fp_df], axis=1)

[14:02:44] WARNING: not removing hydrogen atom without neighbors


In [ ]:
df.columns

Index([        'NR-AR',     'NR-AR-LBD',        'NR-AhR',  'NR-Aromatase',
               'NR-ER',     'NR-ER-LBD', 'NR-PPAR-gamma',        'SR-ARE',
            'SR-ATAD5',        'SR-HSE',
       ...
                  1014,            1015,            1016,            1017,
                  1018,            1019,            1020,            1021,
                  1022,            1023],
      dtype='object', length=1044)

In [ ]:
targets = [
    'NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase',
    'NR-ER','NR-ER-LBD','NR-PPAR-gamma',
    'SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53'
]

y = df[targets]

X = df.drop(columns=targets + ['mol_id', 'smiles'])
X.columns = X.columns.astype(str)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(X.columns[:10])   # just to inspect

X shape: (7823, 1030)
y shape: (7823, 12)
Index(['MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA', 'RotBonds', '0', '1',
       '2', '3'],
      dtype='object')


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    )
)

model.fit(X_train, y_train)

MultiOutputClassifier(estimator=RandomForestClassifier(class_weight='balanced_subsample',
                                                       n_estimators=200,
                                                       n_jobs=-1,
                                                       random_state=42))

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

In [ ]:
from sklearn.metrics import roc_auc_score

print("ROC-AUC Scores:\n")

for i, col in enumerate(targets):
    try:
        prob = y_prob[i][:, 1]
        score = roc_auc_score(y_test[col], prob)
        print(f"{col}: {score:.3f}")
    except:
        print(f"{col}: skipped")

ROC-AUC Scores:

NR-AR: 0.798
NR-AR-LBD: 0.853
NR-AhR: 0.873
NR-Aromatase: 0.756
NR-ER: 0.758
NR-ER-LBD: 0.800
NR-PPAR-gamma: 0.887
SR-ARE: 0.790
SR-ATAD5: 0.867
SR-HSE: 0.809
SR-MMP: 0.857
SR-p53: 0.861


In [ ]:
print(X_test)

        MolWt     LogP  HDonors  HAcceptors    TPSA  RotBonds  0  1  2  3  \
1142   75.067 -0.97030        2           2   63.32         1  0  0  0  0   
4262  404.410  2.87950        3           6   91.68         9  0  1  0  0   
2167  318.373  1.62290        0           4   66.92         5  0  0  0  0   
1678  397.491  1.95120        2           5   89.82         8  0  1  0  1   
3537  226.276  1.38940        2           2   78.76         4  0  1  0  0   
...       ...      ...      ...         ...     ...       ... .. .. .. ..   
3919  602.641 -0.45420        9          12  242.98        10  0  1  0  0   
4323  118.132  1.17940        0           3   35.53         2  0  0  0  0   
7354  338.521 -0.70450        2           3   49.69         3  0  0  0  0   
5763  125.127  0.62928        0           3   50.09         3  0  0  0  0   
5013  622.539  0.65020        5          11  228.51        12  0  0  0  0   

      ...  1014  1015  1016  1017  1018  1019  1020  1021  1022  1023  
114

### prediction pipeline

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator
loop
# fingerprint generator (same as before)
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError("Invalid SMILES string")

    # Descriptors
    desc = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol)
    ]

    # Fingerprint
    fp = fpgen.GetFingerprint(mol)
    fp_array = np.array(fp)

    # Combine
    features = np.concatenate([desc, fp_array])

    return features

def prepare_input(smiles, X_columns):
    features = smiles_to_features(smiles)

    # convert to DataFrame
    input_df = pd.DataFrame([features], columns=X_columns)

    return input_df

def predict_smiles(smiles, model, X_columns, targets):

    input_df = prepare_input(smiles, X_columns)

    preds = model.predict(input_df)
    probs = model.predict_proba(input_df)

    result = {}

    for i, col in enumerate(targets):
        result[col] = {
            "prediction": int(preds[0][i]),
            "probability": float(probs[i][0][1])
        }

    return result

X_columns = X.columns


In [ ]:
smiles = "C1=CC=C(C=C1)O"

output = predict_smiles(smiles, model, X_columns, targets)

print(output)

{'NR-AR': {'prediction': 0, 'probability': 0.0}, 'NR-AR-LBD': {'prediction': 0, 'probability': 0.005}, 'NR-AhR': {'prediction': 0, 'probability': 0.04}, 'NR-Aromatase': {'prediction': 0, 'probability': 0.005}, 'NR-ER': {'prediction': 0, 'probability': 0.045}, 'NR-ER-LBD': {'prediction': 0, 'probability': 0.075}, 'NR-PPAR-gamma': {'prediction': 0, 'probability': 0.005}, 'SR-ARE': {'prediction': 0, 'probability': 0.06}, 'SR-ATAD5': {'prediction': 0, 'probability': 0.03}, 'SR-HSE': {'prediction': 0, 'probability': 0.01}, 'SR-MMP': {'prediction': 0, 'probability': 0.075}, 'SR-p53': {'prediction': 0, 'probability': 0.01}}
